In [1]:
#Importing required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import requests
from fredapi import Fred
import plotly.graph_objects as go
import dash
from dash import html, dcc, Input, Output, callback
import json

with open("../config.json") as f:
    config = json.load(f)

In [2]:
plt.style.use('fivethirtyeight')

fkey = config["api_key"]

#Creating a keyword search function
def fred_search(search_text, api_key=fkey, limit=100):
    url = f"https://api.stlouisfed.org/fred/series/search?search_text={search_text}&api_key={api_key}&file_type=json&limit={limit}"
    response = requests.get(url)
    return pd.DataFrame(response.json()['seriess'])

#Get category id based on series id
def category(series_id, api_key=fkey):
    url = f"https://api.stlouisfed.org/fred/series/categories?series_id={series_id}&api_key={api_key}&file_type=json"
    response = requests.get(url).json()
    return response 

#Get the subcategories of the categories and their names along with category ids
def children(cid, api_key=fkey):
    url = f"https://api.stlouisfed.org/fred/category/children?category_id={cid}&api_key={api_key}&file_type=json"
    response = requests.get(url).json()
    return response

#Get series datapoints for graphing
def values(series_id,api_key = fkey):
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={api_key}&file_type=json"
    response = requests.get(url).json()
    return pd.DataFrame(response['observations'])
#Make pandas dataframe to show all the columns on one line
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)

In [3]:
# Creating FRED object:
fred = Fred(api_key=fkey)

#Simplified series datapoints for graphing
def series(series_id):
    return fred.get_series(series_id)

In [4]:
#Creating a function that does data cleaning on the Series provided given the code
def data_cleaning(code):
    #Get the data from the FRED API
    data  = fred.get_series(code)
    #Convert the data to a pandas DataFrame
    data = pd.DataFrame(data)
    #Convert the index to a datetime object
    data.index = pd.to_datetime(data.index)
    #Convert the index to a column
    data['Date'] = data.index
    # Change the column name to Value
    data.rename(columns={0: 'Value'}, inplace=True)
    #Reorder the columns to have Date first
    data = data[['Date', 'Value']]
    #Convert the date column to a datetime object
    data['Date'] = pd.to_datetime(data['Date'])
    #reset the index
    data.reset_index(drop=True, inplace=True)
    # Drop the rows with missing values in the values column
    data.dropna(subset=['Value'], inplace=True)
    return data

In [2]:
#Reading CSV file: 
df = pd.read_csv('../data-tickers/newOrders.csv')
#Level 0 
TM = data_cleaning('AMTMNO') #Total Manufacturing 

#Level 1
Durable = data_cleaning('DGORDER') #Durable Goods

nonDurable = data_cleaning('AMNMNO')

#Level 2 Under Durable
durablesL2 = df[df['Level']==2]
durablesL2[['FRED API Code','Measure']]
twoAPI = durablesL2['FRED API Code'].tolist()
twoMeasure = durablesL2['Measure'].tolist()
twoData = []
for i in range(0,len(twoAPI)):
    xy = data_cleaning(twoAPI[i])
    twoData.append(go.Scatter(x=xy['Date'],y=xy['Value'],name=twoMeasure[i]))


#Subcomponents of Level 2: 

#Building functions to convert a table into graphs :))
def grapher(var):
    API = var['FRED API Code'].tolist()
    Measure = var['Measure'].tolist()
    data1 = []
    for i in range(0,len(API)):
        clean = data_cleaning(API[i])
        data1.append(go.Scatter(x = clean['Date'],y=clean['Value'],name=Measure[i]))
    graph = go.Figure(data = data1)
    graph = graph.update_layout(xaxis_rangeslider_visible=True)
    return graph
    
def grapher1(var1):
    API = var1['FRED API Code']
    Measure = var1['Measure']
    clean = data_cleaning(API)
    data11 = go.Scatter(x= clean['Date'],y=clean['Value'],name=Measure)
    graph1 = go.Figure(data = data11)
    graph1 = graph1.update_layout(xaxis_rangeslider_visible=True)
    return graph1
    
    



#Subcomponent of Primary Metals[2]
pm = df.iloc[2]
pmGraph = grapher1(pm)

pmSub = df.iloc[3:6]
pmSubGraph = grapher(pmSub)

#Subcomponent of Fabricated Metal Products(None) [6]
fmp = df.iloc[6]
fmpGraph = grapher1(fmp)
#Subcomponent of Machinery [7]
machinery  = df.iloc[7]
machineryGraph = grapher1(machinery)

machinerySub = df.iloc[8:17]
machinerySubGraph = grapher(machinerySub)
#Subcomponent of Computer and Electronic Products[17]
cep = df.iloc[17]
cepGraph = grapher1(cep)

cepSub = df.iloc[18:25]
cepSubGraph = grapher(cepSub)
#Subcomponent of Electrical Equipment, Appliances and Components[25]
eac = df.iloc[25]
eacGraph = grapher1(eac)

eacSub = df.iloc[26:29]
eacSubGraph = grapher(eacSub)
#Subcomponent of Transporation Equipment [29]
te = df.iloc[29]
teGraph = grapher1(te)

teSub = df.iloc[30:34]
teSubGraph = grapher(teSub)
#Subcomponent of Furniture and Related Products(None)[34]
frp = df.iloc[34]
frpGraph = grapher1(frp)

#Testing the code


NameError: name 'data_cleaning' is not defined

In [6]:
#Total manufacturing graph
trace1 = go.Scatter(x=TM['Date'],y=TM['Value'],name='Total Manufacturing')
figTM = go.Figure(data=[trace1])
figTM = figTM.update_layout(xaxis_rangeslider_visible=True)

#Durable and Non-durable Goods (Level 1)

graphDurableNon = go.Scatter(x=nonDurable['Date'],y=nonDurable['Value'],name='Non-Durable Goods')
graphDurable = go.Scatter(x=Durable['Date'],y=Durable['Value'],name = 'Durable Goods')
figDND = go.Figure(data=[graphDurableNon, graphDurable])
figDND = figDND.update_layout(xaxis_rangeslider_visible=True)


#Durable Goods all Subcomponents Graph: 
twoGraph = go.Figure(data=twoData)
twoGraph = twoGraph.update_layout(xaxis_rangeslider_visible=True)


In [7]:
# starting the Dash application: 
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1(children='Dashboard for New Orders'),

    html.Div(children=
        '''Total Manufacturing'''
    ),

    dcc.Graph(
        id='totalManufacturing',
        figure=figTM
    ),
    dcc.Dropdown(id='tmSub', options = [
        {'label':'Show Subcomponents', 'value':'TMS'}
    ], placeholder="Toggle :)"
                ),
    html.Div(id ='TMSS'), html.Div(id='dropdown2'), html.Div(id='Level-2'), html.Div(id='Level-3')
    
])

#Call back function for the first component:
@callback(
    Output('TMSS', 'children'), Output('dropdown2','children'),
    Input('tmSub', 'value')
)
def update_graph(select):
    if select == 'TMS':
        return (dcc.Graph(figure=figDND),
                dcc.Dropdown(id='dropdown_2', options = [
                    {'label':'Non-Durable Goods','value':'NDG'},
                    {'label':'Durable Goods','value':'DG'}
                ],placeholder="Toggle :)")
               )
    else:
        return html.Div(), html.Div()
#Callback function for the level 2 subcomponents
@callback(
    Output('Level-2','children'), Input('dropdown_2','value')
)

def update_graph2(choose):
    if choose =='NDG':
        chosen = go.Figure(data=[graphDurableNon])
        chosen = chosen.update_layout(xaxis_rangeslider_visible=True)
        return dcc.Graph(figure=chosen)
    elif choose == 'DG':
        return (dcc.Graph(figure= twoGraph),
                dcc.Dropdown(id='dropdown_3',options = [
                    {'label':'Primary Metals','value':'Primary Metals'},
                    {'label':'Machinery','value':'Machinery'}
                ],placeholder="Toggle :)")
               )
    else:
        return html.Div()

@callback(
    Output('Level-3','children'), Input('dropdown_3','value')
)

def update3(val):
    if val == 'Primary Metals':
        return dcc.Graph(figure = pmSubGraph)
    elif val == 'Machinery':
        return dcc.Graph(figure = machinerySubGraph)
    


#Run Web application
if __name__ == '__main__':
    app.run(debug=True)

